In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/synthea-cleaned/synthea_clinical_notes_clean.csv
/kaggle/input/synthea/synthea_clinical_notes.csv


In [2]:
# ============================================================
# NOTEBOOK 07 — SYNTHEA LLM INFERENCE (FIXED - ALL 4 MODELS)
# ============================================================

# -----------------------------
# AUTHENTICATION
# -----------------------------
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
from huggingface_hub import login
login(token=secret_value_0)

# -----------------------------
# IMPORTS
# -----------------------------
import os
import gc
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
import warnings
warnings.filterwarnings('ignore')

# -----------------------------
# CONFIG
# -----------------------------
DATA_PATH = "/kaggle/input/synthea-cleaned/synthea_clinical_notes_clean.csv"  # Update this path
OUTPUT_DIR = "/kaggle/working/synthea_predictions"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"

BATCH_SIZE = 4
MAX_NEW_TOKENS = 150
MAX_INPUT_LENGTH = 2048

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Model configurations with proper formatting
MODELS = {
    "mistral": {
        "name": "mistralai/Mistral-7B-Instruct-v0.3",
        "use_chat_template": True,
        "format": "mistral"
    },
    "gemma": {
        "name": "google/gemma-2b-it",
        "use_chat_template": True,
        "format": "gemma"
    },
    "biomistral": {
        "name": "BioMistral/BioMistral-7B",
        "use_chat_template": False,
        "format": "base"
    },
    "qwen": {
        "name": "Qwen/Qwen2.5-7B-Instruct",
        "use_chat_template": True,
        "format": "qwen"
    }
}

# -----------------------------
# PROMPT TEMPLATE
# -----------------------------
BASE_INSTRUCTION = """You are a clinical information extraction system.

TASK: Extract only patient-reported symptoms or clinician-observed findings from the clinical note.

RULES:
- Extract ONLY symptoms and clinical findings
- Do NOT extract: diagnoses, medications, labs, procedures, demographics, family history
- Do NOT include negated symptoms (e.g., "denies chest pain")
- Do NOT infer symptoms not explicitly mentioned
- Output format: One symptom per line, each starting with a hyphen (-)
- If no symptoms found, output exactly: none

Clinical Note:
{note}

Extracted Symptoms:"""

# -----------------------------
# PROMPT FORMATTING
# -----------------------------
def format_prompt(note, model_format, tokenizer):
    """Format prompt according to model's expected template."""
    if model_format == "mistral":
        return f"[INST] {BASE_INSTRUCTION.format(note=note)} [/INST]"
    
    elif model_format == "gemma":
        messages = [{"role": "user", "content": BASE_INSTRUCTION.format(note=note)}]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    elif model_format == "qwen":
        messages = [{"role": "user", "content": BASE_INSTRUCTION.format(note=note)}]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    else:  # base format (BioMistral)
        return BASE_INSTRUCTION.format(note=note)

# -----------------------------
# OUTPUT PARSER
# -----------------------------
def parse_model_output(raw_output):
    """Parse model output to extract symptom list."""
    symptoms = []
    lines = raw_output.strip().split('\n')
    
    for line in lines:
        line = line.strip()
        
        if not line:
            continue
        
        # Skip instruction-like lines
        if any(x in line.lower() for x in ['extracted symptoms:', 'task:', 'rules:', 'clinical note:']):
            continue
        
        # Extract symptom from various list formats
        if line.startswith('-'):
            symptom = line[1:].strip()
        elif line.startswith('•') or line.startswith('*'):
            symptom = line[1:].strip()
        elif len(line) > 2 and line[0].isdigit() and line[1] in '.):':
            symptom = line[2:].strip()
        else:
            # Accept if short and doesn't look like instructions
            if len(line) < 100 and not any(x in line.lower() for x in ['extract', 'output', 'format']):
                symptom = line
            else:
                continue
        
        symptom = symptom.lower().strip().rstrip('.,;:')
        
        if symptom and symptom != 'none' and symptom not in symptoms:
            symptoms.append(symptom)
    
    return symptoms if symptoms else ["none"]

# -----------------------------
# CHECKPOINT MANAGEMENT
# -----------------------------
def save_checkpoint(model_key, processed_indices):
    """Save checkpoint of processed indices."""
    checkpoint_file = f"{CHECKPOINT_DIR}/{model_key}_checkpoint.txt"
    with open(checkpoint_file, 'w') as f:
        f.write(','.join(map(str, processed_indices)))

def load_checkpoint(model_key):
    """Load checkpoint if exists."""
    checkpoint_file = f"{CHECKPOINT_DIR}/{model_key}_checkpoint.txt"
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, 'r') as f:
            content = f.read().strip()
            if content:
                indices = [int(x) for x in content.split(',')]
                print(f"  Resuming from checkpoint: {len(indices)} notes processed")
                return set(indices)
    return set()

# -----------------------------
# LOAD DATA
# -----------------------------
print("Loading Synthea clinical notes...")
df = pd.read_csv(DATA_PATH)
print(f"✓ Loaded {len(df)} notes")

if "note_id" not in df.columns:
    df["note_id"] = range(len(df))

# -----------------------------
# MAIN INFERENCE LOOP
# -----------------------------
for model_key, model_config in MODELS.items():
    print(f"\n{'='*70}")
    print(f"Starting inference for: {model_key.upper()}")
    print(f"Model: {model_config['name']}")
    print(f"{'='*70}")
    
    # Check if already completed
    output_file = f"{OUTPUT_DIR}/{model_key}_predictions.csv"
    if os.path.exists(output_file):
        existing_df = pd.read_csv(output_file)
        if len(existing_df) == len(df):
            print(f"✓ {model_key} already completed ({len(existing_df)} predictions)")
            continue
    
    # Load checkpoint
    processed_indices = load_checkpoint(model_key)
    
    try:
        # Load model
        print(f"\nLoading model...")
        tokenizer = AutoTokenizer.from_pretrained(model_config['name'])
        
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
            tokenizer.pad_token_id = tokenizer.eos_token_id
        
        model = AutoModelForCausalLM.from_pretrained(
            model_config['name'],
            torch_dtype=torch.float16,
            device_map="auto",
            low_cpu_mem_usage=True
        )
        model.eval()
        print(f"✓ Model loaded")
        
        results = []
        notes = df["clinical_note"].tolist()
        note_ids = df["note_id"].tolist()
        
        # Process in batches
        with tqdm(total=len(notes), desc=f"{model_key}") as pbar:
            for batch_idx in range(0, len(notes), BATCH_SIZE):
                batch_end = min(batch_idx + BATCH_SIZE, len(notes))
                batch_notes = notes[batch_idx:batch_end]
                batch_ids = note_ids[batch_idx:batch_end]
                
                # Skip if already processed
                if all(idx in processed_indices for idx in range(batch_idx, batch_end)):
                    pbar.update(len(batch_notes))
                    continue
                
                try:
                    # Format prompts
                    prompts = [format_prompt(note, model_config['format'], tokenizer) 
                              for note in batch_notes]
                    
                    # Tokenize (don't manually set device with device_map="auto")
                    inputs = tokenizer(
                        prompts,
                        return_tensors="pt",
                        padding=True,
                        truncation=True,
                        max_length=MAX_INPUT_LENGTH
                    )
                    
                    # Move inputs to model's device
                    inputs = {k: v.to(model.device) for k, v in inputs.items()}
                    
                    # Generate
                    with torch.no_grad():
                        outputs = model.generate(
                            **inputs,
                            max_new_tokens=MAX_NEW_TOKENS,
                            do_sample=False,
                            temperature=None,
                            top_p=None,
                            pad_token_id=tokenizer.pad_token_id,
                            eos_token_id=tokenizer.eos_token_id
                        )
                    
                    # Decode
                    decoded_outputs = tokenizer.batch_decode(outputs, skip_special_tokens=True)
                    
                    # Parse
                    for i, (decoded, note_id) in enumerate(zip(decoded_outputs, batch_ids)):
                        # Remove prompt from output
                        prompt_len = len(prompts[i])
                        response = decoded[prompt_len:].strip() if len(decoded) > prompt_len else decoded
                        
                        symptoms = parse_model_output(response)
                        
                        results.append({
                            "note_id": note_id,
                            "model": model_key,
                            "predicted_symptoms": str(symptoms),
                            "raw_output": response[:500]
                        })
                        
                        processed_indices.add(batch_idx + i)
                    
                    pbar.update(len(batch_notes))
                    
                    # Checkpoint every 10 batches
                    if (batch_idx // BATCH_SIZE) % 10 == 0:
                        save_checkpoint(model_key, list(processed_indices))
                
                except RuntimeError as e:
                    print(f"\n⚠️  Error in batch {batch_idx}: {e}")
                    pbar.update(len(batch_notes))
                    continue
        
        # Save results
        if results:
            results_df = pd.DataFrame(results)
            results_df.to_csv(output_file, index=False)
            print(f"\n✓ Saved {len(results)} predictions to: {output_file}")
        
        # Cleanup
        del model
        del tokenizer
        torch.cuda.empty_cache()
        gc.collect()
        
        print(f"✓ {model_key} complete\n")
    
    except Exception as e:
        print(f"\n❌ Failed to run {model_key}: {e}")
        continue

# -----------------------------
# SUMMARY
# -----------------------------
print("\n" + "="*70)
print("ALL MODELS COMPLETE")
print("="*70)

print("\nResults Summary:")
for model_key in MODELS.keys():
    output_file = f"{OUTPUT_DIR}/{model_key}_predictions.csv"
    if os.path.exists(output_file):
        df_results = pd.read_csv(output_file)
        print(f"  ✓ {model_key}: {len(df_results)} predictions")
    else:
        print(f"  ✗ {model_key}: No predictions file found")

print("\n✅ Synthea inference pipeline complete!")
print(f"Output directory: {OUTPUT_DIR}")

Loading Synthea clinical notes...
✓ Loaded 500 notes

Starting inference for: MISTRAL
Model: mistralai/Mistral-7B-Instruct-v0.3

Loading model...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✓ Model loaded


qwen:   0%|          | 0/500 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
qwen: 100%|██████████| 500/500 [12:50<00:00,  1.54s/it]



✓ Saved 500 predictions to: /kaggle/working/synthea_predictions/qwen_predictions.csv
✓ qwen complete


ALL MODELS COMPLETE

Results Summary:
  ✓ mistral: 500 predictions
  ✓ gemma: 500 predictions
  ✓ biomistral: 500 predictions
  ✓ qwen: 500 predictions

✅ Synthea inference pipeline complete!
Output directory: /kaggle/working/synthea_predictions
